# 03 · Model Deployment — Cross-Workspace Handoff & Serving

**Team: ML Engineering / Deployment (Workspace B)**

> 🔁 **This notebook runs in a *different workspace* than `01`/`02`.** PCC's training team and
> deployment team work in separate workspaces that share **one Unity Catalog metastore**. Because
> the model and feature table are **governed UC objects**, the deployment team never needs a copy
> of the code, the artifacts, or the training data — they just need the right **UC privileges**.

What this notebook shows:
1. **The handoff** — exactly which grants the training team runs so the deployment team (and the
   serving endpoint) can use the model and its features.
2. **Online store publish** — materialize the feature table to a **Lakebase online store** so
   features can be looked up in real time.
3. **Real-time serving** — deploy the UC model to a **Model Serving endpoint** and score a patient
   by **`patient_id` alone** (features fetched automatically online).
4. Recap of **offline/batch** lookup (already shown in `02`) — the same model, both modes.

In [0]:
%pip install --quiet databricks-feature-engineering==0.13.0.1 "mlflow>=3.8.1" databricks-sdk
%restart_python

## Config
In Workspace B we don't `%run` the training team's setup notebook — instead we point at the
**same UC objects** by name. Set the catalog/schema to match what the training team used
(their derived schema). Everything else is a governed UC reference.

In [0]:
dbutils.widgets.text("catalog", "pcc_mlops_workshop", "Catalog (shared metastore)")
dbutils.widgets.text("schema", "", "Schema used by the training team (their derived schema)")

CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA  = dbutils.widgets.get("schema").strip()
assert SCHEMA, "Set the 'schema' widget to the training team's schema (e.g. jane_doe)."

FEATURE_TABLE = f"{CATALOG}.{SCHEMA}.prth_patient_features"
LABELS_TABLE  = f"{CATALOG}.{SCHEMA}.prth_labels"
MODEL_NAME    = f"{CATALOG}.{SCHEMA}.prth_readmission_lgbm"
PRIMARY_KEY   = "patient_id"
ENDPOINT_NAME = f"prth-readmission-{SCHEMA}".replace("_", "-")
ONLINE_STORE  = f"prth-online-{SCHEMA}".replace("_", "-")          # Lakebase online store (instance) name
ONLINE_TABLE  = f"{CATALOG}.{SCHEMA}.prth_patient_features_online"  # published online copy of the feature table

print(f"Model         : {MODEL_NAME}")
print(f"Feature table : {FEATURE_TABLE}")
print(f"Endpoint      : {ENDPOINT_NAME}")

## Step 1 — The cross-workspace handoff (grants)
These `GRANT`s are run **by the training team (Workspace A)** — the owners of the model and
feature table. Because grants live in the **metastore**, they take effect in **every** workspace
attached to it, including Workspace B. This is the entire handoff: no artifact copy, no export.

The deployment **principal** (the human or service principal creating the endpoint) needs:
- `USE CATALOG`, `USE SCHEMA`
- `EXECUTE` on the model (this is what grants access to the model *artifacts* — no separate access needed)
- `SELECT` on the feature table (to publish it online)

> Replace `deployment_team` with the actual UC group / service principal. Run these **in Workspace A**
> (or here, if you have MANAGE on the objects). They're shown as SQL so PCC can lift them verbatim.

```sql
-- === Run by the TRAINING TEAM (owners) in Workspace A ===
GRANT USE CATALOG ON CATALOG pcc_mlops_workshop                       TO `deployment_team`;
GRANT USE SCHEMA  ON SCHEMA  pcc_mlops_workshop.jane_doe              TO `deployment_team`;
GRANT EXECUTE     ON MODEL   pcc_mlops_workshop.jane_doe.prth_readmission_lgbm TO `deployment_team`;
GRANT SELECT      ON TABLE   pcc_mlops_workshop.jane_doe.prth_patient_features TO `deployment_team`;
```

**What the deployment team does NOT need:** access to the training notebooks, the raw dataset,
MLflow experiment write access, or any storage credential. `EXECUTE ON MODEL` + `USE` on the
namespace is the complete surface for standing up a serving endpoint.

### Verify the handoff worked
From Workspace B, confirm we can *see* the governed objects the training team registered. If
these succeed, the grants are in place and the metastore is shared correctly.

In [0]:
from mlflow.tracking import MlflowClient
import mlflow

mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()

model_version = client.get_model_version_by_alias(MODEL_NAME, "prod")
print(f"✅ Can see model {MODEL_NAME} @prod -> v{model_version.version}")
print(f"✅ Can read feature table: {spark.table(FEATURE_TABLE).count():,} rows")

## Step 2 — Publish the feature table to a Lakebase online store
Offline `score_batch` reads features from the Delta table. **Real-time** serving needs them in a
**low-latency online store** — Databricks' managed **Lakebase** online store. Publishing keeps
the online copy in sync with the offline feature table.

> **Re-run caveat:** the published online table is linked to the offline feature table by its
> internal `table_id`. If you ever **re-run notebook 02** (which drops & recreates the feature
> table), that link goes stale and serving fails with *"No suitable online store found"*. If that
> happens, run the **reset cell below** once, then continue. On a first, clean run you can skip it.

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient
from databricks.sdk import WorkspaceClient

fe = FeatureEngineeringClient()

# --- OPTIONAL RESET (uncomment only if you re-ran notebook 02 and now hit a stale-link error) ---
# Removes the old online store and the orphaned synced online table so the publish below
# re-links to the current feature table. Safe to leave commented on a clean first run.
#
# try:
#     fe.delete_online_store(name=ONLINE_STORE)
# except Exception as e:
#     print("no existing online store to delete:", e)
# try:
#     WorkspaceClient().database.delete_synced_database_table(ONLINE_TABLE)
# except Exception as e:
#     print("no orphaned synced table to drop:", e)

# 2a. Provision (or reuse) a Databricks online store — a managed, Lakebase-backed,
#     low-latency store. This spins up an autoscaling Lakebase instance (takes a few minutes).
#     Note: get_online_store returns None (not an error) when the store doesn't exist yet.
online_store = fe.get_online_store(name=ONLINE_STORE)
if online_store is None:
    online_store = fe.create_online_store(
        name=ONLINE_STORE,
        capacity="CU_1",          # smallest capacity unit; CU_1/CU_2/CU_4/CU_8
    )
    print(f"Created online store {ONLINE_STORE}")
else:
    print(f"Reusing existing online store {ONLINE_STORE}")

# 2a-wait. The online store provisions a Lakebase instance asynchronously — wait for AVAILABLE
#         before publishing (the serving layer can't use a store that isn't ready).
import time
state = "PENDING"
for _ in range(60):
    os_obj = fe.get_online_store(name=ONLINE_STORE)
    state = str(os_obj.state) if os_obj is not None else "PENDING"
    if "AVAILABLE" in state:
        break
    print(f"  online store state = {state} ... waiting")
    time.sleep(20)
print(f"Online store {ONLINE_STORE} is {state}")

# 2b. Publish the offline feature table into the online store. TRIGGERED keeps the online
#     copy in sync from the feature table's Change Data Feed (enabled in notebook 02).
fe.publish_table(
    source_table_name=FEATURE_TABLE,
    online_table_name=ONLINE_TABLE,
    online_store=online_store,
    publish_mode="TRIGGERED",
)
print(f"Published {FEATURE_TABLE} -> {ONLINE_TABLE} (online).")

# 2b-wait. TRIGGERED publish syncs asynchronously. Wait until the online table is fully online
#         so the serving endpoint can discover it during feature-lookup setup.
for _ in range(60):
    detailed = WorkspaceClient().database.get_synced_database_table(
        ONLINE_TABLE
    ).data_synchronization_status.detailed_state
    if "ONLINE_NO_PENDING_UPDATE" in str(detailed):
        break
    print(f"  synced table state = {detailed} ... waiting")
    time.sleep(20)
print(f"Online table {ONLINE_TABLE} is {detailed}")

Created online store prth-online-moe-abdelsamed
  online store state = State.UPDATING ... waiting
Online store prth-online-moe-abdelsamed is State.AVAILABLE
Published pcc_mlops_demo_catalog.moe_abdelsamed.prth_patient_features -> pcc_mlops_demo_catalog.moe_abdelsamed.prth_patient_features_online (online).
  synced table state = SyncedTableState.SYNCED_TABLE_PROVISIONING_PIPELINE_RESOURCES ... waiting
  synced table state = SyncedTableState.SYNCED_TABLE_PROVISIONING_INITIAL_SNAPSHOT ... waiting
Online table pcc_mlops_demo_catalog.moe_abdelsamed.prth_patient_features_online is SyncedTableState.SYNCED_TABLE_ONLINE_NO_PENDING_UPDATE


## Step 3 — Deploy the model to a real-time Model Serving endpoint
We serve the **`@prod`** version — the feature-store-packaged model from notebook `02`. Because
the feature-lookup metadata travels with the model, the endpoint knows to fetch features from the
online store at request time. The endpoint's runtime identity inherits the same UC grants.

In [0]:
from datetime import timedelta
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput

w = WorkspaceClient()

# First-time serving deployments build a container and wire up the online feature lookup,
# which can take 15-30 minutes. Give the waiter enough headroom.
DEPLOY_TIMEOUT = timedelta(minutes=45)

served = ServedEntityInput(
    entity_name=MODEL_NAME,
    entity_version=model_version.version,
    workload_size="Small",             # "Small" | "Medium" | "Large"
    scale_to_zero_enabled=True,
)

existing = [e.name for e in w.serving_endpoints.list()]
if ENDPOINT_NAME in existing:
    print(f"Updating existing endpoint {ENDPOINT_NAME} ...")
    w.serving_endpoints.update_config_and_wait(
        name=ENDPOINT_NAME, served_entities=[served], timeout=DEPLOY_TIMEOUT,
    )
else:
    print(f"Creating endpoint {ENDPOINT_NAME} (first deploy can take 15-30 min) ...")
    w.serving_endpoints.create_and_wait(
        name=ENDPOINT_NAME,
        config=EndpointCoreConfigInput(name=ENDPOINT_NAME, served_entities=[served]),
        timeout=DEPLOY_TIMEOUT,
    )
print(f"✅ Endpoint {ENDPOINT_NAME} ready.")

Creating endpoint prth-readmission-moe-abdelsamed (first deploy can take 15-30 min) ...
✅ Endpoint prth-readmission-moe-abdelsamed ready.


## Step 4 — Real-time scoring by key only
The moment of truth: we send **just a `patient_id`**. The endpoint looks the features up from
the online store and returns a prediction — no feature values in the request payload.

In [0]:
sample_ids = [int(r[0]) for r in spark.table(LABELS_TABLE).select(PRIMARY_KEY).limit(3).collect()]

response = w.serving_endpoints.query(
    name=ENDPOINT_NAME,
    dataframe_records=[{PRIMARY_KEY: pid} for pid in sample_ids],
)
print("Real-time predictions (features fetched automatically online):")
for pid, pred in zip(sample_ids, response.predictions):
    print(f"  patient_id={pid}  ->  {pred}")

Real-time predictions (features fetched automatically online):
  patient_id=100000  ->  0
  patient_id=100001  ->  0
  patient_id=100002  ->  0


## Recap — one model, two lookup modes
| Use case | Where | How it's called | Feature source |
|---|---|---|---|
| **Batch scoring** | Notebook / job (`02`) | `fe.score_batch(model_uri, df_of_keys)` | Offline Delta feature table |
| **Real-time** | Serving endpoint (`03`) | `POST` `{patient_id}` | Lakebase online store |

The **same registered UC model** powers both — that's the payoff of packaging it with
`fe.log_model` and automatic feature lookup.

✅ **Deployed.** Next: **`04_monitoring`** to turn on inference logging and drift dashboards.